# Camada Gold - Consolidação e Disponibilização de Dados Analíticos

## Objetivo

A camada Gold é responsável por consolidar e agregar os dados tratados na camada Silver, transformando-os em informações analíticas prontas para consumo por ferramentas de Business Intelligence e análise de dados.

Nesta etapa são aplicadas regras de negócio e agregações que reduzem o volume de dados e facilitam a geração de indicadores, relatórios e dashboards.

## Fontes de Dados

As tabelas analíticas são geradas a partir das seguintes fontes:

- `silver.pix_transactions`
- `silver.payment_statistics`

## Transformações Aplicadas

### Transações PIX

Consolidação das transações PIX em uma única tabela analítica com granularidade geográfica completa.

Agrupamento por:

- Região
- Estado
- Município
- Ano
- Mês

Métricas calculadas:

- Pagamentos Pessoa Física
- Pagamentos Pessoa Jurídica
- Recebimentos Pessoa Física
- Recebimentos Pessoa Jurídica

Tabela gerada:

- `gold.pix_transactions`

A consolidação em uma única tabela permite a criação de hierarquias geográficas no Power BI, possibilitando análises por Região, Estado ou Município sem necessidade de múltiplas tabelas.

---

### Estatísticas de Meios de Pagamento

Consolidação dos meios de pagamento por:

- Ano
- Mês

Métricas de Quantidade:

- Quantidade de Boletos
- Quantidade de Cheques
- Quantidade de DOC
- Quantidade de PIX
- Quantidade de TEC
- Quantidade de TED

Métricas Financeiras:

- Valor movimentado em Boletos
- Valor movimentado em Cheques
- Valor movimentado em DOC
- Valor movimentado em PIX
- Valor movimentado em TEC
- Valor movimentado em TED

Tabela gerada:

- `gold.payment_per_DT`

## Tabelas Geradas

Após as agregações e regras de negócio, os dados são armazenados nas seguintes tabelas Delta Lake:

- `gold.pix_transactions`
- `gold.payment_per_DT`

## Benefícios da Camada Gold

- Disponibiliza dados prontos para consumo analítico;
- Reduz a necessidade de processamento nas ferramentas de BI;
- Centraliza métricas e indicadores de negócio;
- Melhora o desempenho das consultas e dashboards;
- Facilita análises geográficas através de hierarquias Região → Estado → Município;
- Permite comparações temporais da evolução dos meios de pagamento no Brasil;
- Simplifica a modelagem e manutenção dos dashboards.

In [0]:
from pyspark.sql.functions import sum, col

#Leitura das tabelas da camada Silver
pix = spark.table("silver.pix_transactions")
payment = spark.table("silver.payment_statistics")

#Realização de agregações e finalizando os tratamentos de dados na tabela de pagamentos
result_payment =(
    payment.groupBy( "ano", "mes").agg(
    (sum(col("quantidadeBoleto"))*1000).alias("QtdBoleto"),
    (sum(col("quantidadeCheque"))*1000).alias("QtdCheque"),
    (sum(col("quantidadeDOC"))*1000).alias("QtdDOC"),
    (sum(col("quantidadePix"))*1000).alias("QtdPix"),
    (sum(col("quantidadeTEC"))*1000).alias("QtdTEC"),
    (sum(col("quantidadeTED"))*1000).alias("QtdTED"),
    sum("valorBoleto").alias("ValorBoleto"),
    sum("valorCheque").alias("ValorCheque"),
    sum("valorDOC").alias("ValorDOC"),
    sum("valorPix").alias("ValorPix"),
    sum("valorTEC").alias("ValorTEC"),
    sum("valorTED").alias("ValorTED")
    )
)

#Realização de agregações na tabela de PIX
result_pix = (
    pix
    .groupBy("Regiao", "Estado","Municipio","ano", "mes").agg(
        sum("VL_PagadorPF").alias("PagamentosPF"),
        sum("VL_PagadorPJ").alias("PagamentosPJ"),
        sum("VL_RecebedorPF").alias("RecebimentosPF"),
        sum("VL_RecebedorPJ").alias("RecebimentosPJ")
    )
)

#Salvando os dados na camada Gold
result_pix.write.format("delta").mode("overwrite").saveAsTable("gold.pix_per_Municipio")
print("Tabela gold.pix_transaction criada")

result_payment.write.format("delta").mode("overwrite").saveAsTable("gold.payment_per_DT")
print("Tabela gold.payment_transaction criada")